In [8]:
"""
data/dataset.py
===============
KiTS21 dataset.  Returns:
  - train mode : 16-slice tumour-centred crop  (D=16, H, W)
  - val   mode : full volume                   (D,    H, W)

Each sample always includes event flag and survival time for downstream
survival training.
"""

import json
import os
import random

import numpy as np
import SimpleITK as sitk
import torch
import torchvision.transforms.functional as TF
from torch.utils.data import Dataset
from torchvision.transforms import InterpolationMode


class KitsDataset(Dataset):
    def __init__(
        self,
        rootdir:        str,
        target_spacing: tuple,          # (X, Y, Z) SimpleITK convention
        target_shape:   tuple,          # (H, W) after spatial resize
        split_file:     str,
        metadata_path:  str,
        p:              float = 0.8,    # prob of tumour-centred depth crop
        mode:           str   = "train",
        crop_depth:     int   = 16,
    ):
        self.rootdir        = rootdir
        self.target_spacing = target_spacing
        self.target_shape   = target_shape
        self.split_file     = split_file
        self.metadata_path  = metadata_path
        self.p              = p
        self.mode           = mode
        self.crop_depth     = crop_depth

        self.metadata: dict = {}
        self._load_cases()

    # ── Initialisation ────────────────────────────────────────────────────

    def _load_cases(self) -> None:
        with open(self.split_file, "r") as f:
            splits = json.load(f)
        self.cases = splits.get(self.mode, splits.get("train"))

        if self.metadata_path is not None:
            with open(self.metadata_path, "r") as f:
                for entry in json.load(f):
                    self.metadata[entry["case_id"]] = entry

    # ── Dataset protocol ──────────────────────────────────────────────────

    def __len__(self) -> int:
        return len(self.cases)

    def __getitem__(self, index: int) -> dict:
        caseid = self.cases[index]

        image = sitk.ReadImage(os.path.join(self.rootdir, caseid, "imaging.nii.gz"))
        mask  = sitk.ReadImage(os.path.join(self.rootdir, caseid, "aggregated_MAJ_seg.nii.gz"))
        # mask  = sitk.ReadImage(os.path.join(self.rootdir, caseid, "segmentation.nii.gz"))
        image = sitk.DICOMOrient(image, "RAS")
        mask  = sitk.DICOMOrient(mask,  "RAS")

        image, mask = self._resample(image, mask)
        image, mask = self._to_tensors(image, mask)
        image, mask = self._spatial_resize(image, mask)  # H, W → target_shape

        if self.mode == "train":
            image, mask = self._train_crop(image, mask)  # (crop_depth, H, W)
        # val: full volume (D, H, W) — sliding window handled in training loop

        event, survival_time = self._get_survival(caseid)

        return {
            "ct":            image,
            "mask":          mask,
            "caseid":        caseid,
            "event":         torch.tensor(event,         dtype=torch.bool),
            "survival_time": torch.tensor(survival_time, dtype=torch.float32),
        }

    # ── Private helpers ───────────────────────────────────────────────────

    def _resample(
        self,
        image: sitk.Image,
        mask:  sitk.Image,
    ) -> tuple[sitk.Image, sitk.Image]:
        original_size    = image.GetSize()
        original_spacing = image.GetSpacing()

        new_size = [
            int(round(osz * osp / tsp))
            for osz, osp, tsp in zip(original_size, original_spacing, self.target_spacing)
        ]

        def _do_resample(itk_img: sitk.Image, is_mask: bool) -> sitk.Image:
            r = sitk.ResampleImageFilter()
            r.SetSize(new_size)
            r.SetOutputSpacing(self.target_spacing)
            r.SetOutputOrigin(itk_img.GetOrigin())
            r.SetOutputDirection(itk_img.GetDirection())
            if is_mask:
                r.SetInterpolator(sitk.sitkNearestNeighbor)
                r.SetDefaultPixelValue(0)
            else:
                r.SetInterpolator(sitk.sitkLinear)
                r.SetDefaultPixelValue(-1000)
            return r.Execute(itk_img)

        return _do_resample(image, False), _do_resample(mask, True)

    def _to_tensors(
        self,
        image: sitk.Image,
        mask:  sitk.Image,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        img_arr  = sitk.GetArrayFromImage(image).astype(np.float32)  # (D, H, W)
        mask_arr = sitk.GetArrayFromImage(mask).astype(np.int64)      # (D, H, W)

        # HU windowing → [0, 1]
        img_arr = np.clip(img_arr, -200.0, 300.0)
        img_arr = (img_arr + 200.0) / 500.0

        return torch.from_numpy(img_arr), torch.from_numpy(mask_arr)

    def _spatial_resize(
        self,
        image: torch.Tensor,   # (D, H, W)
        mask:  torch.Tensor,   # (D, H, W) int64
    ) -> tuple[torch.Tensor, torch.Tensor]:
        image = TF.resize(
            image.unsqueeze(0),        # (1, D, H, W)
            list(self.target_shape),
            interpolation=InterpolationMode.BILINEAR,
            antialias=True,
        ).squeeze(0)

        mask = TF.resize(
            mask.float().unsqueeze(0),
            list(self.target_shape),
            interpolation=InterpolationMode.NEAREST,
        ).squeeze(0).long()

        return image, mask

    def _train_crop(
        self,
        image: torch.Tensor,
        mask:  torch.Tensor,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        depth = image.shape[0]
        crop  = self.crop_depth

        nonzero_slices = (mask != 0).any(dim=(1, 2))
        indices        = torch.where(nonzero_slices)[0]

        if len(indices) > 0 and torch.rand(()) < self.p:
            center = indices[random.randint(0, len(indices) - 1)].item()
            z_min  = max(0, center - crop + 1)
            z_max  = min(center, depth - crop)
            z = (
                max(0, min(center, depth - crop))
                if z_max < z_min
                else random.randint(z_min, z_max)
            )
        else:
            z = random.randint(0, max(0, depth - crop))

        return image[z:z + crop], mask[z:z + crop]

    def _get_survival(self, caseid: str) -> tuple[bool, float]:
        """
        Returns
        -------
        event         : True if patient died (vital_status == 'dead')
        survival_time : days after surgery (vital_days_after_surgery)
        """
        if self.metadata_path is not None:
            meta          = self.metadata[caseid]
            event         = meta["vital_status"] == "dead"
            survival_time = meta.get("vital_days_after_surgery") or 0.0
            return bool(event), float(survival_time)

RGB_HU_WINDOWS = [
    (-150.0, 250.0),   # R — soft tissue / renal parenchyma
    ( -50.0, 300.0),   # G — tumour / vascular enhancement
    (   0.0, 400.0),   # B — denser structures / late-phase enhancement
]
 
 
class KitsDatasetRGB(KitsDataset):
    """
    3-channel HU-windowed CT dataset for the RGB-OmniRad survival experiment.
 
    Design
    ------
    - Always returns the **full volume** regardless of mode — no tumour-centred
      or random depth cropping.  The full slice sequence is passed to OmniRad
      once to extract features; gated attention pooling handles variable depth.
    - The segmentation mask is **never loaded** — no cropping means it is not
      needed at all, saving I/O time.
    - Each CT is converted to 3 channels by applying three independent HU
      windows and rescaling each to [0, 1].
 
    HU windows (R, G, B)
    --------------------
      R : [-150, 250]  — soft tissue / renal parenchyma
      G : [ -50, 300]  — tumour blush / vascular enhancement
      B : [   0, 400]  — denser structures / late-phase enhancement
 
    Return schema
    -------------
    {"ct_rgb": (D, 3, H, W) float32, "caseid", "event", "survival_time"}
    """
 
    HU_WINDOWS = RGB_HU_WINDOWS   # override at class level if needed
 
    # ── Full item pipeline ────────────────────────────────────────────────
 
    def __getitem__(self, index: int) -> dict:
        caseid = self.cases[index]
 
        # Load and orient image only — mask is not needed (no cropping)
        image = sitk.ReadImage(os.path.join(self.rootdir, caseid, "imaging.nii.gz"))
        image = sitk.DICOMOrient(image, "RAS")
        image = self._resample_image_only(image)
 
        ct_rgb = self._to_rgb_tensor(image)           # (D, 3, H, W)
        ct_rgb = self._spatial_resize_rgb(ct_rgb)     # (D, 3, target_H, target_W)
 
        event, survival_time = self._get_survival(caseid)
 
        return {
            "ct_rgb":        ct_rgb,
            "caseid":        caseid,
            "event":         torch.tensor(event,         dtype=torch.bool),
            "survival_time": torch.tensor(survival_time, dtype=torch.float32),
        }
 
    # ── Resample image only (no mask) ─────────────────────────────────────
 
    def _resample_image_only(self, image: sitk.Image) -> sitk.Image:
        original_size    = image.GetSize()
        original_spacing = image.GetSpacing()
        new_size = [
            int(round(osz * osp / tsp))
            for osz, osp, tsp in zip(original_size, original_spacing, self.target_spacing)
        ]
        r = sitk.ResampleImageFilter()
        r.SetSize(new_size)
        r.SetOutputSpacing(self.target_spacing)
        r.SetOutputOrigin(image.GetOrigin())
        r.SetOutputDirection(image.GetDirection())
        r.SetInterpolator(sitk.sitkLinear)
        r.SetDefaultPixelValue(-1000)
        return r.Execute(image)
 
    # ── 3-channel HU windowing ────────────────────────────────────────────
 
    def _to_rgb_tensor(self, image: sitk.Image) -> torch.Tensor:
        """
        Convert raw HU values to a (D, 3, H, W) float32 tensor.
 
        Each channel is independently clipped to its HU window and rescaled
        to [0, 1].  No global clipping is applied before windowing.
        """
        hu = sitk.GetArrayFromImage(image).astype(np.float32)  # (D, H, W)
 
        channels = []
        for hu_min, hu_max in self.HU_WINDOWS:
            ch = np.clip(hu, hu_min, hu_max)
            ch = (ch - hu_min) / (hu_max - hu_min)
            channels.append(ch)
 
        return torch.from_numpy(np.stack(channels, axis=1))  # (D, 3, H, W)
 
    # ── Spatial resize ────────────────────────────────────────────────────
 
    def _spatial_resize_rgb(self, ct_rgb: torch.Tensor) -> torch.Tensor:
        """Resize (D, 3, H, W) → (D, 3, target_H, target_W). D treated as batch."""
        return TF.resize(
            ct_rgb,
            list(self.target_shape),
            interpolation = InterpolationMode.BILINEAR,
            antialias     = True,
        )
 
    # ── Disable parent methods that don't apply here ─────────────────────
 
    def _to_tensors(self, image, mask):
        raise NotImplementedError("KitsDatasetRGB uses _to_rgb_tensor instead.")
 
    def _spatial_resize(self, image, mask):
        raise NotImplementedError("KitsDatasetRGB uses _spatial_resize_rgb instead.")
 
    def _train_crop(self, image, mask):
        raise NotImplementedError("KitsDatasetRGB never crops — full volume always returned.")
   

In [7]:
# from sklearn.model_selection import train_test_split
# import os
# import json

# ROOT_DIR = "/home/sandeep/kits23/dataset"
# cases = os.listdir(ROOT_DIR)
# X_train, X_test = train_test_split(cases, test_size=0.2)
# print(len(X_test))
# print(len(X_train))
# outputjson = "train_test_kits23.json"
# split = {"train": X_train, "val": X_test}
# with open(outputjson, "w") as f:
#     json.dump(split, f, indent=4)

In [9]:
from configs.survival_config import SurvivalConfig
import os
cfg    = SurvivalConfig()
cfg.root_dir = "/home/sandeep/HECKTOR2025/HECKTOR_2025_Training_Data/Task 1"
cfg.json_path = "/home/sandeep/HECKTOR2025/HECKTOR_2025_Training_Data/dataset_split_fixed.json"
_ds_kwargs = dict(
        rootdir        = cfg.root_dir,
        target_spacing = cfg.target_spacing,
        target_shape   = cfg.target_shape,
        split_file     = cfg.json_path,
        metadata_path = None
        # metadata_path  = os.path.join(cfg.root_dir, "kits23.json"),
    )
train_ds = KitsDataset(**_ds_kwargs, mode="val")

In [10]:
import numpy as np
from torch.utils.data import DataLoader

# Assuming train_ds is already initialized
train_loader = DataLoader(train_ds, batch_size=1, shuffle=True)

# Fetch a single random batch
batch = next(iter(train_loader))

# Note: The shape is now (3, 16, 256, 256) due to the 3 RGB channels
ct_tensor = batch["ct"][0].numpy()       
mask_tensor = batch["mask"][0].numpy()   # Shape: (16, 256, 256)
case_id = batch["caseid"][0]

print(f"Loaded Case: {case_id}")
print(f"CT Tensor Shape: {ct_tensor.shape}")
print(f"Mask Tensor Shape: {mask_tensor.shape}")


TypeError: object of type 'NoneType' has no len()

In [7]:
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

def visualize_kits_batch(ct, mask, case_id):
    # ct: (D, H, W), mask: (D, H, W)
    depth = ct.shape[0]

    @widgets.interact(z=widgets.IntSlider(min=0, max=depth-1, step=1, value=depth//2, description='Slice Z'))
    def plot_slice(z):
        plt.figure(figsize=(12, 6))
        
        # 1. CT Scan Slice
        plt.subplot(1, 2, 1)
        plt.imshow(ct[z], cmap='gray')
        plt.title(f"CT Slice {z} - Case: {case_id}")
        plt.axis('off')
        
        # 2. Overlay (CT + Mask)
        plt.subplot(1, 2, 2)
        plt.imshow(ct[z], cmap='gray')
        
        # Mask overlay: we use a masked array to make 0 (background) transparent
        # Adjust alpha for transparency level
        masked_data = np.ma.masked_where(mask[z] == 0, mask[z])
        plt.imshow(masked_data, cmap='jet', alpha=0.5, interpolation='nearest')
        
        plt.title("Mask Overlay (Tumor/Kidney)")
        plt.axis('off')
        plt.tight_layout()
        plt.show()

# Run the visualizer
visualize_kits_batch(ct_tensor, mask_tensor, case_id)

interactive(children=(IntSlider(value=99, description='Slice Z', max=197), Output()), _dom_classes=('widget-in…

In [3]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from matplotlib.backends.backend_pdf import PdfPages

def load_and_preprocess_data(json_filepath):
    """Loads the JSON metadata and flattens nested dictionaries."""
    with open(json_filepath, 'r') as f:
        data = json.load(f)
    
    df = pd.json_normalize(data)
    print(f"Dataset loaded: {df.shape[0]} rows, {df.shape[1]} columns.")
    return df

def add_text_page(pdf, title, text_content):
    """Helper function to render raw text onto a PDF page."""
    fig = plt.figure(figsize=(8.5, 11))
    plt.axis('off')
    
    fig.text(0.05, 0.95, title, fontsize=14, weight='bold', va='top')
    fig.text(0.05, 0.90, text_content, fontsize=9, family='monospace', va='top')
    
    pdf.savefig(fig)
    plt.close(fig)

def generate_unified_pdf(df, pdf_filepath):
    """Generates a single PDF containing text summaries followed by visualizations."""
    
    with PdfPages(pdf_filepath) as pdf:
        
        # --- PAGE 1: Data Types and Missing Values ---
        info_df = pd.DataFrame({
            'Data Type': df.dtypes,
            'Missing Values': df.isnull().sum(),
            '% Missing': (df.isnull().sum() / len(df)) * 100
        })
        info_str = f"Dataset Shape: {df.shape[0]} rows x {df.shape[1]} columns\n\n"
        info_str += info_df.head(45).to_string() 
        if len(info_df) > 45:
            info_str += "\n... (truncated for space)"
        add_text_page(pdf, "Data Types and Missing Values", info_str)

        # --- PAGE 2: Summary Statistics (Numerical) ---
        num_desc = df.describe().T
        num_desc['variance'] = df.select_dtypes(include=[np.number]).var()
        num_str = num_desc[['count', 'mean', 'std', 'variance', 'min', 'max']].to_string()
        add_text_page(pdf, "Summary Statistics (Numerical)", num_str)

        # --- PAGE 3: Summary Statistics (Categorical) ---
        cat_cols = df.select_dtypes(include=['object', 'bool']).columns
        cat_str = ""
        for col in cat_cols[:12]:
            cat_str += f"Value counts for {col}:\n"
            cat_str += df[col].value_counts(dropna=False).head(5).to_string()
            cat_str += "\n" + "-" * 30 + "\n"
        add_text_page(pdf, "Summary Statistics (Categorical)", cat_str)

        # --- PAGE 4: Histograms ---
        numerical_cols_to_plot = [
            'age_at_nephrectomy', 'bmi', 'vital_days_after_surgery', 
            'operative_time', 'pathologic_size', 'last_preop_egfr.value'
        ]
        numerical_cols_to_plot = [col for col in numerical_cols_to_plot if col in df.columns]

        fig, axes = plt.subplots(2, 3, figsize=(11, 8.5))
        fig.suptitle('Distributions of Key Numerical Variables', fontsize=16)
        
        for i, col in enumerate(numerical_cols_to_plot):
            row, col_idx = divmod(i, 3)
            sns.histplot(df[col].dropna(), kde=True, ax=axes[row, col_idx], color='skyblue')
            axes[row, col_idx].set_title(f'{col}', fontsize=10)
            axes[row, col_idx].set_xlabel('')
            axes[row, col_idx].set_ylabel('Freq')
            
        plt.tight_layout()
        pdf.savefig(fig)
        plt.close(fig)

        # --- PAGE 5: Bar Charts (ERROR FIXED HERE) ---
        categorical_cols_to_plot = [
            'gender', 'surgery_type', 'malignant', 
            'tumor_histologic_subtype', 'aua_risk_score'
        ]
        categorical_cols_to_plot = [col for col in categorical_cols_to_plot if col in df.columns]

        fig, axes = plt.subplots(2, 3, figsize=(11, 8.5))
        fig.suptitle('Counts of Key Categorical Variables', fontsize=16)
        
        for i, col in enumerate(categorical_cols_to_plot):
            row, col_idx = divmod(i, 3)
            
            # FIX: Fill NaNs and convert booleans to strings to prevent sorting crashes
            plot_series = df[col].fillna('Missing').astype(str)
            
            sns.countplot(
                y=plot_series, 
                hue=plot_series, 
                ax=axes[row, col_idx], 
                palette='viridis', 
                order=plot_series.value_counts().index, 
                legend=False
            )
            axes[row, col_idx].set_title(f'{col}', fontsize=10)
            axes[row, col_idx].set_xlabel('')
            axes[row, col_idx].set_ylabel('')
            
        if len(categorical_cols_to_plot) < 6:
            fig.delaxes(axes[1, 2])
            
        plt.tight_layout()
        pdf.savefig(fig)
        plt.close(fig)

        # --- PAGE 6: Correlation Heatmap ---
        fig = plt.figure(figsize=(11, 8.5))
        numeric_df = df.select_dtypes(include=[np.number])
        numeric_df = numeric_df.loc[:, numeric_df.std() > 0]
        
        if not numeric_df.empty:
            corr_matrix = numeric_df.corr()
            mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
            
            sns.heatmap(corr_matrix, mask=mask, cmap='coolwarm', annot=False, 
                        linewidths=0.5, cbar_kws={"shrink": .8})
            plt.title('Correlation Heatmap of Numerical Features', fontsize=16)
            plt.tight_layout()
            pdf.savefig(fig)
            plt.close(fig)
            
    print(f"\nUnified PDF successfully generated: {pdf_filepath}")

if __name__ == "__main__":
    FILE_PATH = '/home/sandeep/kits23/dataset/kits23.json' 
    OUTPUT_PDF = 'kits_unified_eda_report.pdf'
    
    try:
        kits_df = load_and_preprocess_data(FILE_PATH)
        generate_unified_pdf(kits_df, OUTPUT_PDF)
        
    except FileNotFoundError:
        print(f"Error: Could not find '{FILE_PATH}'. Please check the directory.")

Dataset loaded: 489 rows, 63 columns.

Unified PDF successfully generated: kits_unified_eda_report.pdf


In [4]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.backends.backend_pdf import PdfPages
import textwrap
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
import warnings

warnings.filterwarnings('ignore') # Suppress seaborn warnings for cleaner output

# ==========================================
# 1. DATA LOADING & PREPROCESSING
# ==========================================

def load_data(json_filepath):
    """Loads the JSON metadata and flattens nested dictionaries."""
    with open(json_filepath, 'r') as f:
        data = json.load(f)
    return pd.json_normalize(data)

def preprocess_and_track(df, missing_threshold=0.40):
    """
    Cleans data, builds the pipeline, and returns original and processed DataFrames.
    """
    df_original = df.copy()
    
    # Identify missing fractions
    missing_frac = df.isnull().mean()
    cols_to_drop = missing_frac[missing_frac > missing_threshold].index.tolist()
    missing_percentages = (missing_frac[cols_to_drop] * 100).to_dict()
    
    df_cleaned = df.drop(columns=cols_to_drop)

    # Extract Targets
    event = (df_cleaned['vital_status'].str.lower() == 'dead').astype(int).values
    time = df_cleaned['vital_days_after_surgery'].values
    time = np.clip(time, a_min=1e-5, a_max=None) # Ensure strictly positive T
    
    cols_to_exclude = ['vital_status', 'vital_days_after_surgery', 'case_id']
    X = df_cleaned.drop(columns=[col for col in cols_to_exclude if col in df_cleaned.columns])

    # Categorize Features
    numeric_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
    categorical_features = X.select_dtypes(include=['object', 'bool']).columns.tolist()

    # Pipelines
    numeric_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])

    categorical_transformer = Pipeline(steps=[
        ('to_string', FunctionTransformer(lambda x: x.astype(str), validate=False)), 
        ('imputer', SimpleImputer(strategy='constant', fill_value='Missing')),
        ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ('num', numeric_transformer, numeric_features),
            ('cat', categorical_transformer, categorical_features)
        ])

    X_processed_array = preprocessor.fit_transform(X)
    
    cat_encoder = preprocessor.named_transformers_['cat'].named_steps['onehot']
    encoded_cat_names = cat_encoder.get_feature_names_out(categorical_features)
    all_feature_names = numeric_features + list(encoded_cat_names)

    df_processed = pd.DataFrame(X_processed_array, columns=all_feature_names)
    
    return df_original, df_processed, numeric_features, categorical_features, missing_percentages

# ==========================================
# 2. PDF GENERATION LOGIC
# ==========================================

def add_text_page(pdf, title, text_content):
    """Renders wrapped, formatted text onto a PDF page."""
    fig = plt.figure(figsize=(8.5, 11))
    plt.axis('off')
    
    fig.text(0.08, 0.92, title, fontsize=16, weight='bold', va='top')
    
    wrapped_lines = []
    for line in text_content.split('\n'):
        if len(line) > 90:
            wrapped_lines.extend(textwrap.wrap(line, width=90))
        else:
            wrapped_lines.append(line)
            
    wrapped_text = "\n".join(wrapped_lines)
    fig.text(0.08, 0.88, wrapped_text, fontsize=10, family='monospace', va='top', ha='left')
    
    pdf.savefig(fig)
    plt.close(fig)

def plot_continuous_feature(pdf, df_orig, df_proc, col_name):
    """Plots before/after distributions and stats for a continuous feature."""
    fig = plt.figure(figsize=(8.5, 11))
    
    # 1. Title
    fig.text(0.5, 0.95, f"Continuous Feature: {col_name}", fontsize=14, weight='bold', ha='center')
    
    # 2. Plots (Top Half)
    ax1 = fig.add_axes([0.1, 0.55, 0.35, 0.3]) # [left, bottom, width, height]
    ax2 = fig.add_axes([0.55, 0.55, 0.35, 0.3])
    
    sns.histplot(df_orig[col_name].dropna(), kde=True, ax=ax1, color='lightcoral')
    ax1.set_title('Original Data (Raw Scale)')
    ax1.set_ylabel('Frequency')
    
    sns.histplot(df_proc[col_name], kde=True, ax=ax2, color='mediumseagreen')
    ax2.set_title('Processed (Median Imputed & Scaled)')
    ax2.set_ylabel('')
    
    # 3. Statistics (Bottom Half)
    stats_text = f"""
    STATISTICAL SUMMARY:
    ------------------------------------------------------
    Original Data (Before Processing):
    - Missing Values : {df_orig[col_name].isnull().sum()}
    - Mean           : {df_orig[col_name].mean():.2f}
    - Std Dev        : {df_orig[col_name].std():.2f}
    - Min            : {df_orig[col_name].min():.2f}
    - Max            : {df_orig[col_name].max():.2f}
    
    Processed Data (After Standardization):
    - Missing Values : {df_proc[col_name].isnull().sum()}
    - Mean           : {df_proc[col_name].mean():.6f} (Approaching 0)
    - Std Dev        : {df_proc[col_name].std():.6f} (Approaching 1)
    - Min            : {df_proc[col_name].min():.2f}
    - Max            : {df_proc[col_name].max():.2f}
    
    TRANSFORMATION PIPELINE:
    1. Median Imputation: Replaced missing values with the median to resist outliers.
    2. Standard Scaling : Z = (x - mean) / std. 
       This ensures the neural network treats this feature's scale equally 
       relative to other features.
    """
    fig.text(0.1, 0.45, stats_text, fontsize=10, family='monospace', va='top', ha='left')
    
    pdf.savefig(fig)
    plt.close(fig)

def plot_discrete_feature(pdf, df_orig, df_proc, col_name):
    """Plots before/after counts and stats for a discrete/categorical feature."""
    fig = plt.figure(figsize=(8.5, 11))
    
    # 1. Title
    fig.text(0.5, 0.95, f"Discrete Feature: {col_name}", fontsize=14, weight='bold', ha='center')
    
    # 2. Get OHE columns associated with this original categorical feature
    ohe_cols = [c for c in df_proc.columns if c.startswith(f"{col_name}_")]
    
    # 3. Plots
    ax1 = fig.add_axes([0.1, 0.55, 0.35, 0.3])
    ax2 = fig.add_axes([0.55, 0.55, 0.35, 0.3])
    
    # Original Plot
    plot_series = df_orig[col_name].fillna('Missing (NaN)').astype(str)
    sns.countplot(y=plot_series, ax=ax1, palette='viridis', order=plot_series.value_counts().index)
    ax1.set_title('Original Data Counts')
    ax1.set_xlabel('Count')
    ax1.set_ylabel('')
    
    # Processed Plot (Sum of 1s in OHE columns)
    ohe_sums = df_proc[ohe_cols].sum().sort_values(ascending=False)
    # Strip prefix for cleaner labels
    clean_labels = [label.replace(f"{col_name}_", "") for label in ohe_sums.index]
    
    sns.barplot(x=ohe_sums.values, y=clean_labels, ax=ax2, palette='mako')
    ax2.set_title('Processed (One-Hot Encoded)')
    ax2.set_xlabel('Count (Sum of 1s)')
    ax2.set_ylabel('')
    
    # 4. Statistics
    stats_text = f"""
    STATISTICAL SUMMARY & TRANSFORMATION:
    ------------------------------------------------------
    Original Data Missing Values: {df_orig[col_name].isnull().sum()}
    
    One-Hot Encoding Expansion:
    The original column '{col_name}' was expanded into {len(ohe_cols)} binary vectors.
    """
    
    for label, count in zip(clean_labels, ohe_sums.values):
         stats_text += f"\n- {col_name}_{label:<20} : {int(count)} instances (1s)"
         
    stats_text += """\n
    TRANSFORMATION PIPELINE:
    1. Constant Imputation: NaN values were explicitly replaced with 'Missing'.
    2. One-Hot Encoding: Converted string labels into orthogonal binary vectors.
       This allows the ML model to assign distinct weights to each category 
       without assuming any mathematical order (e.g., Male is not 'greater' 
       than Female).
    """
    
    fig.text(0.1, 0.45, stats_text, fontsize=10, family='monospace', va='top', ha='left')
    
    pdf.savefig(fig)
    plt.close(fig)

# ==========================================
# 3. MASTER REPORT GENERATOR
# ==========================================

def generate_report(df_orig, df_proc, num_cols, cat_cols, dropped_dict, pdf_filepath):
    with PdfPages(pdf_filepath) as pdf:
        
        # --- PAGE 1: Feature Types ---
        page1_text = "CONTINUOUS (NUMERICAL) FEATURES:\n" + "-"*40 + "\n"
        page1_text += "\n".join([f"- {col}" for col in num_cols]) + "\n\n"
        
        page1_text += "DISCRETE (CATEGORICAL) FEATURES:\n" + "-"*40 + "\n"
        page1_text += "\n".join([f"- {col}" for col in cat_cols])
        
        add_text_page(pdf, "1. Variable Classification", page1_text)

        # --- PAGE 2: Dropped Columns ---
        page2_text = "COLUMNS DROPPED DUE TO EXCESSIVE MISSINGNESS (>40%)\n" + "-"*60 + "\n\n"
        if not dropped_dict:
            page2_text += "No columns exceeded the missingness threshold.\n"
        else:
            for col, pct in dropped_dict.items():
                page2_text += f"- {col} : {pct:.2f}% Missing\n"
        
        page2_text += """\n
        MACHINE LEARNING JUSTIFICATION:
        In Survival Analysis and Neural Networks, excessive missingness introduces critical flaws:
        1. Noise Injection: When a column is >40% missing, imputation (guessing the value) 
           dominates the actual signal.
        2. Spurious Correlations: The model may learn the 'imputation rule' rather than 
           finding a real physiological link to patient survival.
        3. Multi-Modal Balance: Because you are fusing this tabular data with highly 
           dense OmniRad image features, sparse/noisy tabular columns can destabilize 
           the MLP gradients. Dropping them forces the network to rely on clean data.
        """
        add_text_page(pdf, "2. Data Trimming & Feature Selection", page2_text)

        # --- PAGES 3+: Continuous Features (Select subset for EDA brevity) ---
        # Limit to top 6 continuous features to keep PDF concise
        selected_num_cols = [c for c in ['age_at_nephrectomy', 'bmi', 'operative_time', 'pathologic_size', 'estimated_blood_loss', 'hospitalization'] if c in num_cols]
        for col in selected_num_cols:
            plot_continuous_feature(pdf, df_orig, df_proc, col)
            
        # --- PAGES N+: Discrete Features (Select subset for EDA brevity) ---
        # Limit to top 6 discrete features
        selected_cat_cols = [c for c in ['gender', 'surgery_type', 'malignant', 'tumor_histologic_subtype', 'aua_risk_score', 'clavien_surgical_complications'] if c in cat_cols]
        for col in selected_cat_cols:
            plot_discrete_feature(pdf, df_orig, df_proc, col)

    print(f"\nReport successfully generated: {pdf_filepath}")

# ==========================================
# EXECUTION
# ==========================================
if __name__ == "__main__":
    FILE_PATH = '/home/sandeep/RAW_DATA/kits23/dataset/kits23.json' 
    OUTPUT_PDF = 'Complete_Preprocessing_Report.pdf'
    
    try:
        print("Loading data...")
        raw_df = load_data(FILE_PATH)
        
        print("Executing transformation pipeline...")
        df_orig, df_proc, num_cols, cat_cols, dropped_dict = preprocess_and_track(raw_df, missing_threshold=0.40)
        
        print(f"Generating PDF with charts and statistical tables...")
        generate_report(df_orig, df_proc, num_cols, cat_cols, dropped_dict, OUTPUT_PDF)
        
    except FileNotFoundError:
        print(f"Error: Could not find '{FILE_PATH}'. Please verify the path.")

Loading data...
Executing transformation pipeline...
Generating PDF with charts and statistical tables...

Report successfully generated: Complete_Preprocessing_Report.pdf
